In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [5]:
organism = "homo_sapiens"
cell_source = "ovary"
cell_state = None
table_name = "clean_degs.xlsx" # local name

## Get sheet names

In [9]:
metadata = pd.read_json("../metadata.json")
sheet_names = [obj["sheet_name"] for obj in metadata["tables"]]

sheet_names

['Variable genes clusters']

## Read in file

In [12]:
excel = pd.read_excel(table_name, sheet_name = sheet_names, skiprows = 2)

for e in list(excel.values()):
    print(e.columns)

Index(['gene', 'p_val', 'avg_logFC', 'pct.1', 'pct.2', 'p_val_adj', 'cluster'], dtype='object')


In [13]:
cols = [
    {
        "sheet_name": sheet_name,
        "cols": {
            "avg_logFC": "logfc",
            "gene": "feature_name",
            "p_val": "p_val",
            "p_val_adj": "p_corr",
            "cluster": "cell_type_label"
        }
    } for sheet_name in sheet_names]

In [14]:
for idx, sheet in enumerate(cols):
    sname = sheet["sheet_name"]
    print(sname)
    excel_name = excel
    if idx < 2:
        excel_name = excel
    # only keep the columns we want
    excel_name[sname] = excel_name[sname].loc[:,sheet["cols"].keys()].rename(columns=sheet["cols"])
    # add sheet name to the dataframe
    excel_name[sname]["sheet_name"] = sname

Variable genes clusters


In [15]:
df = pd.concat(list(excel.values()))
df.head()

,logfc,feature_name,p_val,p_corr,cell_type_label,sheet_name
0,2.388708,FIGLA,0.0,0.0,oocytes,Variable genes clusters
1,2.335580,KPNA7,0.0,0.0,oocytes,Variable genes clusters
2,2.243391,NLRP5,0.0,0.0,oocytes,Variable genes clusters
3,2.106073,C15orf60,0.0,0.0,oocytes,Variable genes clusters
4,2.085036,ZAR1,0.0,0.0,oocytes,Variable genes clusters


## Unfiltered

In [16]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None 

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({
        "extracted": {
            "organism": row["organism"], 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene"
            },
        "derived": {
            "organism": row["organism"], 
            "cell_type_id": None, 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene", 
            "feature_identifier": row["feature_identifier"], 
            "feature_type": "ensembl"
            },
        "source": {
            "source_type": "deg", 
            "source_rationale": "unfiltered", 
            "source_id": row["sheet_name"],
            "source_metrics": {
                "p_corr": row["p_corr"],
                "log_fc": row["logfc"],
            }
            }
        })

result[:1]

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'oocytes',
   'cell_source': 'ovary',
   'cell_state': None,
   'feature_name': 'FIGLA',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'oocytes',
   'cell_source': 'ovary',
   'cell_state': None,
   'feature_name': 'FIGLA',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'Variable genes clusters',
   'source_metrics': {'p_corr': 0.0, 'log_fc': 2.38870825301472}}}]

## Save unfiltered

In [18]:
with open("evidence_unfiltered_metrics.json", 'w') as f:
    json.dump(result, f, indent=4)

## Perform the filtering

In [12]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "cluster"
col_rank = "pct.1"

In [13]:
min_mean = 15
max_pval = 0.05
min_lfc = 1.5
max_gene_shares = 10
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [14]:
markers[col_ct].value_counts()

cluster
perivascular cells    20
monocytes             20
oocytes               20
endothelial cells     20
t cells               20
granulosa cells       10
stroma                 1
Name: count, dtype: int64

In [15]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
granulosa cells       0.78230
perivascular cells    0.79420
oocytes               0.81235
endothelial cells     0.83625
monocytes             0.83640
t cells               0.91450
stroma                0.96300
Name: pct.1, dtype: float64

## Convert to evidence json



In [19]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

In [20]:
save

[{'cell_type_label': 'granulosa cells',
  'gene': 'HSPA6',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000173110'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'PHLDA1',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000139289'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'NDUFA4L2',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000185633'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'RRAD',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000166592'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'MT1M',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': None,
  'gene_id': 'ENSG00000205364'},
 {'cell_type_label': 'perivascular cells',
  'gene': 'CRIP1',
  'organism': 'homo_sapiens',
  'cell_source': 'ovary',
  'cell_state': No

## Save evidence

In [21]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 